# 🔥 Databricks – Working with Spark SQL & Streaming Ingestion
### Topics Covered
| # | Topic |
|---|-------|
| 1 | Spark SQL – Joins, Aggregations & Window Functions |
| 2 | Dates & Timestamps in Spark SQL |
| 3 | Auto Loader (`cloudFiles`) |
| 4 | MERGE INTO – Upsert Pattern |

> **Level:** Intermediate | **Runtime:** Databricks 13.x+ (DBR ML or Standard) | **Author:** Raamashaamy


---
## 📘 Section 1 – Spark SQL: Joins, Aggregations & Window Functions

We will use a realistic e-commerce dataset — **orders**, **customers**, and **products** — to practise joins, aggregations and window functions in Spark SQL.


### 1.1 – Create Sample Tables

In [0]:
%sql
-- ============================================================
-- Drop & recreate a demo database (safe for re-runs)
-- ============================================================
CREATE DATABASE IF NOT EXISTS demo_sparksql;
USE demo_sparksql;

-- Customers
CREATE OR REPLACE TABLE customers (
  customer_id INT,
  customer_name STRING,
  city STRING,
  segment STRING
);

INSERT INTO customers VALUES
  (1, 'Arjun Sharma',   'Chennai',   'Retail'),
  (2, 'Priya Menon',    'Bengaluru', 'Corporate'),
  (3, 'Ravi Kumar',     'Hyderabad', 'Retail'),
  (4, 'Sneha Iyer',     'Mumbai',    'Corporate'),
  (5, 'Kiran Das',      'Pune',      'SMB');

-- Products
CREATE OR REPLACE TABLE products (
  product_id INT,
  product_name STRING,
  category STRING,
  unit_price DOUBLE
);

INSERT INTO products VALUES
  (101, 'Laptop',    'Electronics', 75000.0),
  (102, 'Phone',     'Electronics', 25000.0),
  (103, 'Desk',      'Furniture',   12000.0),
  (104, 'Chair',     'Furniture',    8000.0),
  (105, 'Headphones','Electronics',  3500.0);

-- Orders
CREATE OR REPLACE TABLE orders (
  order_id     INT,
  customer_id  INT,
  product_id   INT,
  order_date   DATE,
  quantity     INT,
  discount_pct DOUBLE
);

INSERT INTO orders VALUES
  (1001, 1, 101, '2024-01-05', 2, 0.05),
  (1002, 2, 102, '2024-01-12', 1, 0.00),
  (1003, 1, 105, '2024-02-03', 3, 0.10),
  (1004, 3, 103, '2024-02-14', 1, 0.00),
  (1005, 4, 101, '2024-03-01', 1, 0.05),
  (1006, 2, 104, '2024-03-10', 4, 0.00),
  (1007, 5, 102, '2024-03-22', 2, 0.10),
  (1008, 1, 103, '2024-04-07', 1, 0.00),
  (1009, 3, 105, '2024-04-15', 5, 0.05),
  (1010, 4, 102, '2024-04-28', 2, 0.00);


### 1.2 – INNER, LEFT & ANTI Joins

In [0]:
%sql
-- ── INNER JOIN ─────────────────────────────────────────────
-- Revenue per order (joining all three tables)
SELECT
  o.order_id,
  c.customer_name,
  p.product_name,
  p.category,
  o.quantity,
  p.unit_price,
  o.discount_pct,
  ROUND(o.quantity * p.unit_price * (1 - o.discount_pct), 2) AS net_revenue
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
INNER JOIN products  p ON o.product_id  = p.product_id
ORDER BY o.order_id;


In [0]:
%sql
-- ── LEFT JOIN ──────────────────────────────────────────────
-- All customers, even those with no orders yet
SELECT
  c.customer_id,
  c.customer_name,
  c.segment,
  COUNT(o.order_id) AS total_orders
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name, c.segment
ORDER BY total_orders DESC;


In [0]:
%sql
-- ── ANTI JOIN ──────────────────────────────────────────────
-- Products that have never been ordered
SELECT p.product_id, p.product_name, p.category
FROM products p
LEFT JOIN orders o ON p.product_id = o.product_id
WHERE o.order_id IS NULL;


### 1.3 – Aggregations

In [0]:
%sql
-- Revenue & order count by customer segment + product category
SELECT
  c.segment,
  p.category,
  COUNT(o.order_id)                                              AS order_count,
  SUM(o.quantity * p.unit_price * (1 - o.discount_pct))         AS gross_revenue,
  AVG(o.quantity * p.unit_price * (1 - o.discount_pct))         AS avg_order_value,
  MAX(o.quantity * p.unit_price * (1 - o.discount_pct))         AS max_order_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products  p ON o.product_id  = p.product_id
GROUP BY c.segment, p.category
ORDER BY gross_revenue DESC;


In [0]:
%sql
-- HAVING: only segments with gross revenue > 100,000
SELECT
  c.segment,
  ROUND(SUM(o.quantity * p.unit_price * (1 - o.discount_pct)), 2) AS total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products  p ON o.product_id  = p.product_id
GROUP BY c.segment
HAVING total_revenue > 100000
ORDER BY total_revenue DESC;


### 1.4 – Window Functions

In [0]:
%sql
-- ── RANK & DENSE_RANK ──────────────────────────────────────
-- Top customers by revenue per segment
WITH order_revenue AS (
  SELECT
    c.customer_id,
    c.customer_name,
    c.segment,
    ROUND(SUM(o.quantity * p.unit_price * (1 - o.discount_pct)), 2) AS total_revenue
  FROM orders o
  JOIN customers c ON o.customer_id = c.customer_id
  JOIN products  p ON o.product_id  = p.product_id
  GROUP BY c.customer_id, c.customer_name, c.segment
)
SELECT *,
  RANK()       OVER (PARTITION BY segment ORDER BY total_revenue DESC) AS rnk,
  DENSE_RANK() OVER (PARTITION BY segment ORDER BY total_revenue DESC) AS dense_rnk
FROM order_revenue
ORDER BY segment, rnk;


In [0]:
%sql
-- ── RUNNING TOTAL & MOVING AVERAGE ────────────────────────
-- Monthly running total of revenue (ordered by month)
WITH monthly AS (
  SELECT
    DATE_FORMAT(o.order_date, 'yyyy-MM')                             AS order_month,
    ROUND(SUM(o.quantity * p.unit_price * (1 - o.discount_pct)), 2) AS monthly_revenue
  FROM orders o
  JOIN products p ON o.product_id = p.product_id
  GROUP BY order_month
)
SELECT
  order_month,
  monthly_revenue,
  ROUND(SUM(monthly_revenue)  OVER (ORDER BY order_month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_total,
  ROUND(AVG(monthly_revenue)  OVER (ORDER BY order_month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2)         AS moving_avg_3m
FROM monthly
ORDER BY order_month;


In [0]:
%sql
-- ── LAG / LEAD ─────────────────────────────────────────────
-- Month-over-month revenue change per category
WITH cat_monthly AS (
  SELECT
    p.category,
    DATE_FORMAT(o.order_date, 'yyyy-MM')                             AS order_month,
    ROUND(SUM(o.quantity * p.unit_price * (1 - o.discount_pct)), 2) AS revenue
  FROM orders o
  JOIN products p ON o.product_id = p.product_id
  GROUP BY p.category, order_month
)
SELECT
  category,
  order_month,
  revenue,
  LAG(revenue)  OVER (PARTITION BY category ORDER BY order_month) AS prev_month_revenue,
  ROUND(revenue - LAG(revenue) OVER (PARTITION BY category ORDER BY order_month), 2) AS mom_change
FROM cat_monthly
ORDER BY category, order_month;


---
## 📘 Section 2 – Dates & Timestamps in Spark SQL

Spark SQL has a rich set of date/time functions. This section covers the most commonly used ones in a data engineering context.


### 2.1 – Core Date Functions

In [0]:
%sql
-- ── Current date / time ───────────────────────────────────
SELECT
  current_date()                          AS today,
  current_timestamp()                     AS now,
  date_format(current_timestamp(), 'dd-MMM-yyyy HH:mm:ss') AS formatted_now;


In [0]:
%sql
-- ── Extraction ────────────────────────────────────────────
SELECT
  order_date,
  year(order_date)       AS yr,
  month(order_date)      AS mo,
  day(order_date)        AS dy,
  dayofweek(order_date)  AS dow,   -- 1=Sunday
  weekofyear(order_date) AS woy,
  quarter(order_date)    AS qtr
FROM demo_sparksql.orders
ORDER BY order_date;


In [0]:
%sql
-- ── Arithmetic ────────────────────────────────────────────
SELECT
  order_date,
  date_add(order_date, 30)              AS plus_30_days,
  date_sub(order_date, 7)               AS minus_7_days,
  datediff(current_date(), order_date)  AS days_since_order,
  add_months(order_date, 3)             AS plus_3_months
FROM demo_sparksql.orders
ORDER BY order_date;


### 2.2 – String ↔ Date Conversions

In [0]:
%sql
-- to_date / to_timestamp / unix_timestamp
SELECT
  '2024-03-15'                                             AS raw_string,
  to_date('2024-03-15', 'yyyy-MM-dd')                      AS parsed_date,
  to_timestamp('2024-03-15 14:30:00', 'yyyy-MM-dd HH:mm:ss') AS parsed_ts,
  unix_timestamp('2024-03-15', 'yyyy-MM-dd')               AS epoch_secs,
  from_unixtime(unix_timestamp('2024-03-15','yyyy-MM-dd'))  AS back_to_string;


### 2.3 – Timestamp & Timezone Handling

In [0]:
%sql
-- ── Truncation ────────────────────────────────────────────
SELECT
  current_timestamp()                         AS ts,
  date_trunc('hour',  current_timestamp())    AS trunc_hour,
  date_trunc('day',   current_timestamp())    AS trunc_day,
  date_trunc('month', current_timestamp())    AS trunc_month,
  date_trunc('year',  current_timestamp())    AS trunc_year;


In [0]:
%sql
-- ── Timezone conversion ───────────────────────────────────
SELECT
  current_timestamp()                                      AS utc_ts,
  from_utc_timestamp(current_timestamp(), 'Asia/Kolkata')  AS ist_ts,
  to_utc_timestamp(current_timestamp(), 'Asia/Kolkata')    AS back_to_utc;


### 2.4 – Real-World Pattern: Late-Arriving Data Window

A common data engineering task — filter records that fall inside a 30-minute processing window.


In [0]:
%sql
-- Simulate an events table with event_time and processing_time
CREATE OR REPLACE TABLE demo_sparksql.events (
  event_id       INT,
  event_time     TIMESTAMP,
  processing_time TIMESTAMP,
  payload        STRING
);

INSERT INTO demo_sparksql.events VALUES
  (1, '2024-04-01 10:00:00', '2024-04-01 10:02:00', 'click'),
  (2, '2024-04-01 10:00:00', '2024-04-01 10:35:00', 'click'),   -- late!
  (3, '2024-04-01 10:01:00', '2024-04-01 10:03:00', 'view'),
  (4, '2024-04-01 09:55:00', '2024-04-01 10:28:00', 'purchase'); -- borderline

SELECT *,
  (unix_timestamp(processing_time) - unix_timestamp(event_time)) / 60  AS delay_minutes,
  CASE
    WHEN (unix_timestamp(processing_time) - unix_timestamp(event_time)) <= 1800
    THEN 'within_window'
    ELSE 'late_arriving'
  END AS arrival_status
FROM demo_sparksql.events;


---
## 📘 Section 3 – Auto Loader (`cloudFiles`)

Auto Loader incrementally ingests new files arriving in cloud storage (ADLS, S3, GCS) into Delta tables. It uses **file notifications** or **directory listing** to detect new files — far more efficient than COPY INTO for large-scale continuous ingestion.

### How it works
```
Cloud Storage (ADLS/S3)
       │  new files land
       ▼
  Auto Loader (cloudFiles source)
  ├── tracks processed files in a checkpoint
  ├── schema inference & evolution built-in
       │
       ▼
  Delta Table (Bronze layer)
```


### 3.1 – Simulating a Landing Zone (Batch Files)

In [0]:
# ── Python cell ──
# In a real environment these would be files landing in ADLS/S3.
# We simulate them in DBFS for the Community Edition / sandbox.

import json
from datetime import datetime, timedelta

dbutils.fs.mkdirs("/tmp/autoloader_demo/landing")
dbutils.fs.mkdirs("/tmp/autoloader_demo/checkpoint")
dbutils.fs.mkdirs("/tmp/autoloader_demo/schema")

# Write two batches of JSON files
batch_1 = [
    {"order_id": 2001, "customer_id": 1, "product_id": 101, "amount": 75000, "event_ts": "2024-05-01T09:00:00"},
    {"order_id": 2002, "customer_id": 3, "product_id": 102, "amount": 25000, "event_ts": "2024-05-01T09:05:00"},
    {"order_id": 2003, "customer_id": 2, "product_id": 105, "amount": 10500, "event_ts": "2024-05-01T09:10:00"},
]

batch_2 = [
    {"order_id": 2004, "customer_id": 4, "product_id": 103, "amount": 12000, "event_ts": "2024-05-02T10:00:00"},
    {"order_id": 2005, "customer_id": 5, "product_id": 104, "amount": 32000, "event_ts": "2024-05-02T10:15:00"},
]

for i, record in enumerate(batch_1):
    path = f"/dbfs/tmp/autoloader_demo/landing/batch1_file{i}.json"
    with open(path, "w") as f:
        json.dump(record, f)

print("✅ Batch 1 files written to landing zone")
print("Files:", dbutils.fs.ls("/tmp/autoloader_demo/landing"))


### 3.2 – Auto Loader Stream (Trigger Once / Available Now)

In [0]:
# ── Python cell ──
# readStream with cloudFiles format
# trigger(availableNow=True) processes all pending files then stops —
# ideal for scheduled batch-style ingestion.

from pyspark.sql.functions import current_timestamp, input_file_name

df_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", "/tmp/autoloader_demo/schema")
         .option("cloudFiles.inferColumnTypes", "true")   # infer INT / TIMESTAMP etc.
         .load("/tmp/autoloader_demo/landing")
)

# Add audit columns (best practice in Bronze layer)
df_bronze = (
    df_stream
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file",  input_file_name())
)

query = (
    df_bronze.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", "/tmp/autoloader_demo/checkpoint")
             .trigger(availableNow=True)               # process all pending, then stop
             .toTable("demo_sparksql.bronze_orders_raw")
)

query.awaitTermination()
print("✅ Batch 1 ingested into bronze_orders_raw")


In [0]:
%sql
-- Verify what landed in Bronze
SELECT *, _source_file, _ingested_at
FROM demo_sparksql.bronze_orders_raw
ORDER BY order_id;


### 3.3 – Incremental Load: New Files Arrive

In [0]:
# ── Python cell ──
# Write Batch 2 files — simulating new data arriving AFTER the first run

import json

for i, record in enumerate(batch_2):
    path = f"/dbfs/tmp/autoloader_demo/landing/batch2_file{i}.json"
    with open(path, "w") as f:
        json.dump(record, f)

print("✅ Batch 2 files written. Auto Loader will only pick up NEW files.")


In [0]:
# ── Python cell ──
# Re-run the same stream — checkpoint ensures only new files are processed

query2 = (
    df_bronze.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", "/tmp/autoloader_demo/checkpoint")
             .trigger(availableNow=True)
             .toTable("demo_sparksql.bronze_orders_raw")
)

query2.awaitTermination()
print("✅ Batch 2 incremental load complete")


In [0]:
%sql
-- Should now show all 5 records with different _source_file values
SELECT order_id, customer_id, amount, _source_file
FROM demo_sparksql.bronze_orders_raw
ORDER BY order_id;


### 3.4 – Auto Loader Key Options Cheatsheet

| Option | Purpose | Common Value |
|--------|---------|--------------|
| `cloudFiles.format` | File format | `json`, `csv`, `parquet`, `avro` |
| `cloudFiles.schemaLocation` | Where to persist inferred schema | A DBFS/ADLS path |
| `cloudFiles.inferColumnTypes` | Infer proper types (not just STRING) | `true` |
| `cloudFiles.schemaEvolutionMode` | What to do when new columns arrive | `addNewColumns` (default) |
| `cloudFiles.useNotifications` | Use cloud event notifications (prod) | `true` |
| `trigger(availableNow=True)` | Process all pending files, then stop | Scheduled jobs |
| `trigger(processingTime="5 minutes")` | Micro-batch every N minutes | Near-real-time |


---
## 📘 Section 4 – MERGE INTO (Upsert Pattern)

`MERGE INTO` is the cornerstone of Silver-layer processing in the Medallion Architecture. It allows you to atomically handle **inserts**, **updates**, and **deletes** in a single statement against a Delta table.

### Why MERGE over simple overwrite?
- Preserves history for unchanged records
- Handles CDC (Change Data Capture) streams naturally
- Works with Delta's ACID guarantees — no partial writes


### 4.1 – Create a Silver Target Table

In [0]:
%sql
-- This is our "source of truth" Silver orders table
CREATE OR REPLACE TABLE demo_sparksql.silver_orders (
  order_id     INT,
  customer_id  INT,
  product_id   INT,
  order_date   DATE,
  quantity     INT,
  discount_pct DOUBLE,
  status       STRING,
  updated_at   TIMESTAMP
);

-- Seed with initial data (already "processed" orders)
INSERT INTO demo_sparksql.silver_orders VALUES
  (1001, 1, 101, '2024-01-05', 2, 0.05, 'confirmed', current_timestamp()),
  (1002, 2, 102, '2024-01-12', 1, 0.00, 'confirmed', current_timestamp()),
  (1003, 1, 105, '2024-02-03', 3, 0.10, 'confirmed', current_timestamp());

SELECT * FROM demo_sparksql.silver_orders ORDER BY order_id;


### 4.2 – Basic MERGE: Insert + Update

In [0]:
%sql
-- Incoming data from Bronze (new orders + updates to existing ones)
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
  -- UPDATE: order 1001 quantity changed from 2 → 3
  (1001, 1, 101, DATE'2024-01-05', 3, 0.05, 'updated'),
  -- UPDATE: order 1002 is now cancelled
  (1002, 2, 102, DATE'2024-01-12', 1, 0.00, 'cancelled'),
  -- INSERT: brand new orders
  (1004, 3, 103, DATE'2024-02-14', 1, 0.00, 'confirmed'),
  (1005, 4, 101, DATE'2024-03-01', 1, 0.05, 'confirmed')
AS t(order_id, customer_id, product_id, order_date, quantity, discount_pct, status);

-- ── MERGE ─────────────────────────────────────────────────
MERGE INTO demo_sparksql.silver_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
  UPDATE SET
    target.quantity     = source.quantity,
    target.discount_pct = source.discount_pct,
    target.status       = source.status,
    target.updated_at   = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (order_id, customer_id, product_id, order_date, quantity, discount_pct, status, updated_at)
  VALUES (source.order_id, source.customer_id, source.product_id, source.order_date,
          source.quantity, source.discount_pct, source.status, current_timestamp());


In [0]:
%sql
-- Verify: order 1001 quantity=3, order 1002 cancelled, new rows 1004 & 1005 added
SELECT * FROM demo_sparksql.silver_orders ORDER BY order_id;


### 4.3 – MERGE with DELETE (Full CDC Pattern)

In [0]:
%sql
-- CDC-style source: each record has an operation flag (I=Insert, U=Update, D=Delete)
CREATE OR REPLACE TEMP VIEW cdc_feed AS
SELECT * FROM VALUES
  ('U', 1003, 1, 105, DATE'2024-02-03', 5, 0.10, 'confirmed'),   -- update qty
  ('D', 1004, 3, 103, DATE'2024-02-14', 1, 0.00, 'confirmed'),   -- delete this row
  ('I', 1006, 2, 104, DATE'2024-03-10', 4, 0.00, 'confirmed')    -- new insert
AS t(op, order_id, customer_id, product_id, order_date, quantity, discount_pct, status);

MERGE INTO demo_sparksql.silver_orders AS target
USING cdc_feed AS source
ON target.order_id = source.order_id
WHEN MATCHED AND source.op = 'D' THEN DELETE
WHEN MATCHED AND source.op = 'U' THEN
  UPDATE SET
    target.quantity   = source.quantity,
    target.status     = source.status,
    target.updated_at = current_timestamp()
WHEN NOT MATCHED AND source.op = 'I' THEN
  INSERT (order_id, customer_id, product_id, order_date, quantity, discount_pct, status, updated_at)
  VALUES (source.order_id, source.customer_id, source.product_id, source.order_date,
          source.quantity, source.discount_pct, source.status, current_timestamp());


In [0]:
%sql
-- order 1003 qty should be 5, order 1004 deleted, order 1006 inserted
SELECT * FROM demo_sparksql.silver_orders ORDER BY order_id;


### 4.4 – Conditional MERGE (Insert-Only if Not Exists)

In [0]:
%sql
-- Common pattern: deduplicate — only insert if record doesn't exist
MERGE INTO demo_sparksql.silver_orders AS target
USING (
  SELECT 1007 AS order_id, 5 AS customer_id, 102 AS product_id,
         DATE'2024-03-22' AS order_date, 2 AS quantity,
         0.10 AS discount_pct, 'confirmed' AS status
) AS source
ON target.order_id = source.order_id
WHEN NOT MATCHED THEN
  INSERT (order_id, customer_id, product_id, order_date, quantity, discount_pct, status, updated_at)
  VALUES (source.order_id, source.customer_id, source.product_id, source.order_date,
          source.quantity, source.discount_pct, source.status, current_timestamp());


### 4.5 – MERGE via PySpark (Python API)

In [0]:
# ── Python cell ──
# DeltaTable API — preferred when building reusable pipeline functions

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Source DataFrame (could come from Auto Loader, readStream, etc.)
source_df = spark.createDataFrame([
    (1001, 1, 101, "2024-01-05", 2, 0.05, "shipped"),
    (1008, 1, 103, "2024-04-07", 1, 0.00, "confirmed"),
], ["order_id","customer_id","product_id","order_date","quantity","discount_pct","status"])

source_df = source_df.withColumn("order_date", F.to_date("order_date"))

# Load the Delta target
dt = DeltaTable.forName(spark, "demo_sparksql.silver_orders")

(
    dt.alias("target")
      .merge(source_df.alias("source"), "target.order_id = source.order_id")
      .whenMatchedUpdate(set={
          "target.quantity":     "source.quantity",
          "target.status":       "source.status",
          "target.updated_at":   F.expr("current_timestamp()")
      })
      .whenNotMatchedInsertAll()   # insert all columns from source when no match
      .execute()
)

spark.table("demo_sparksql.silver_orders").orderBy("order_id").show()


### 4.6 – MERGE Best Practices

| Practice | Why |
|----------|-----|
| Always match on a **unique business key** (`order_id`) | Prevents duplicate processing |
| Add `updated_at = current_timestamp()` in UPDATE | Tracks when the record last changed |
| Use `WHEN NOT MATCHED BY SOURCE THEN DELETE` cautiously | Deletes any row not in incoming batch (full replace semantics) |
| Combine with Auto Loader | Natural Bronze → Silver pipeline |
| Use `foreachBatch` in Structured Streaming | Apply MERGE inside a streaming job |


---
## 📘 Section 5 – End-to-End Mini Pipeline

Combining Auto Loader + MERGE to build a Bronze → Silver pipeline.

```
JSON files (landing zone)
        │
        ▼  Auto Loader (cloudFiles)
   Bronze Table  (raw, append-only)
        │
        ▼  foreachBatch + MERGE INTO
   Silver Table  (deduplicated, current state)
        │
        ▼  Aggregation Query
   Gold View     (business metrics)
```


In [0]:
# ── Python cell ──
# Silver table already exists from Section 4.
# This demonstrates using foreachBatch to apply MERGE inside a stream.

from delta.tables import DeltaTable
from pyspark.sql import functions as F

def upsert_to_silver(micro_batch_df, batch_id):
    """
    Called by foreachBatch for each micro-batch.
    Applies MERGE so the Silver table always reflects current state.
    """
    # Cast types from raw JSON
    micro_batch_df = (
        micro_batch_df
        .withColumn("order_id",     F.col("order_id").cast("int"))
        .withColumn("customer_id",  F.col("customer_id").cast("int"))
        .withColumn("product_id",   F.col("product_id").cast("int"))
        .withColumn("amount",       F.col("amount").cast("double"))
        .withColumn("event_ts",     F.to_timestamp("event_ts"))
        .withColumn("status",       F.lit("confirmed"))
        .withColumn("updated_at",   F.current_timestamp())
    )

    dt = DeltaTable.forName(spark, "demo_sparksql.silver_orders")

    (
        dt.alias("target")
          .merge(micro_batch_df.alias("source"), "target.order_id = source.order_id")
          .whenMatchedUpdate(set={
              "target.updated_at": "source.updated_at"
          })
          .whenNotMatchedInsert(values={
              "target.order_id":     "source.order_id",
              "target.customer_id":  "source.customer_id",
              "target.product_id":   "source.product_id",
              "target.order_date":   F.expr("date(source.event_ts)"),
              "target.quantity":     F.lit(1),
              "target.discount_pct": F.lit(0.0),
              "target.status":       "source.status",
              "target.updated_at":   "source.updated_at",
          })
          .execute()
    )
    print(f"✅ Batch {batch_id} merged into silver_orders")


# Read from Bronze Auto Loader stream
bronze_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", "/tmp/autoloader_demo/schema")
         .load("/tmp/autoloader_demo/landing")
)

pipeline_query = (
    bronze_stream.writeStream
                 .foreachBatch(upsert_to_silver)
                 .option("checkpointLocation", "/tmp/autoloader_demo/pipeline_checkpoint")
                 .trigger(availableNow=True)
                 .start()
)

pipeline_query.awaitTermination()
print("✅ Pipeline complete — Silver table updated")


In [0]:
%sql
-- Gold: final revenue report combining all sections
SELECT
  c.segment,
  p.category,
  COUNT(o.order_id)                                       AS orders,
  ROUND(SUM(o.quantity * p.unit_price * (1-o.discount_pct)), 2) AS total_revenue,
  RANK() OVER (ORDER BY SUM(o.quantity * p.unit_price * (1-o.discount_pct)) DESC) AS revenue_rank
FROM demo_sparksql.silver_orders o
JOIN demo_sparksql.customers c  ON o.customer_id = c.customer_id
JOIN demo_sparksql.products  p  ON o.product_id  = p.product_id
GROUP BY c.segment, p.category
ORDER BY revenue_rank;


---
## 🧹 Cleanup (Optional — Run after training)

In [0]:
%sql
DROP DATABASE IF EXISTS demo_sparksql CASCADE;


In [0]:
dbutils.fs.rm("/tmp/autoloader_demo", recurse=True)
print("✅ Cleanup done")


---
## ✅ Summary

| Section | Key Takeaway |
|---------|-------------|
| **Joins** | INNER, LEFT, ANTI — always think about what happens to unmatched rows |
| **Aggregations** | GROUP BY + HAVING; use CTEs to keep queries readable |
| **Window Functions** | RANK, DENSE_RANK, LAG/LEAD, running totals — no GROUP BY needed |
| **Dates** | `date_format`, `date_add`, `datediff`, `date_trunc`, `from_utc_timestamp` |
| **Auto Loader** | Incremental ingestion via `cloudFiles`; checkpoint ensures exactly-once |
| **MERGE INTO** | Handles Insert + Update + Delete atomically on Delta tables |
| **Pipeline** | Auto Loader → `foreachBatch` + MERGE = robust Bronze→Silver pattern |

> **Next steps:** Delta Live Tables (DLT) automates this entire pipeline declaratively — no manual `foreachBatch` needed!
